# 03 — Analyze Visium H&E Data

**Spatial Transcriptomics Project | 10x Genomics**

This notebook applies Squidpy for full spatial analysis of Visium H&E data, going beyond image features into spatial statistics and molecular interactions.

**Dataset:** Mouse brain coronal section — Visium H&E stained, pre-annotated clusters.

**H&E vs Fluorescence:** H&E (Hematoxylin & Eosin) stains all cells non-specifically — it is the gold standard in clinical pathology. Fluorescence (notebook 02) uses antibody-specific channels. H&E enables broader tissue morphology analysis.

**Steps:** Load → spatial clusters → image features → neighborhood enrichment → co-occurrence → ligand-receptor → Moran's I

## 1. Install Dependencies

In [ ]:
# leidenalg is required by older scanpy leiden calls inside cluster_features
!pip install squidpy anndata scanpy igraph leidenalg -q

## 2. Import Packages & Load Data

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import squidpy as sq

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

In [ ]:
# Load pre-processed Visium H&E dataset (downloads automatically)
img = sq.datasets.visium_hne_image()
adata = sq.datasets.visium_hne_adata()

print(adata)
print(f"Spots: {adata.n_obs} | Genes: {adata.n_vars}")
print(f"Clusters: {adata.obs['cluster'].unique().tolist()}")

## 3. Visualize Spatial Cluster Annotation

H&E tissue with Leiden clusters from gene expression overlaid.

> **Save this plot**

In [ ]:
sq.pl.spatial_scatter(adata, color="cluster")
plt.savefig("visium_hne_cluster_spatial.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: visium_hne_cluster_spatial.png")

## 4. Extract Multi-Scale Image Features

Summary features at scale=1.0 (spot region) and scale=2.0 (more surrounding context). Higher scale = richer morphological context.

In [ ]:
for scale in [1.0, 2.0]:
    feature_name = f"features_summary_scale{scale}"
    print(f"Calculating features at scale={scale}...")
    sq.im.calculate_image_features(
        adata,
        img.compute(),
        features="summary",
        key_added=feature_name,
        n_jobs=4,
        scale=scale,
    )

# Combine across scales
adata.obsm["features"] = pd.concat(
    [adata.obsm[f] for f in adata.obsm.keys() if "features_summary" in f],
    axis="columns",
)
adata.obsm["features"].columns = ad.utils.make_index_unique(
    adata.obsm["features"].columns
)

print(f"Combined features shape: {adata.obsm['features'].shape}")

## 5. Cluster Image Features & Compare to Gene Clusters

> **Save this plot**

In [ ]:
def cluster_features(features: pd.DataFrame, like=None):
    """
    Leiden clustering on image features.
    Uses igraph backend to avoid leidenalg import errors.
    """
    if like is not None:
        features = features.filter(like=like)

    tmp = ad.AnnData(features)
    sc.pp.scale(tmp)
    sc.pp.pca(tmp, n_comps=min(10, features.shape[1] - 1))
    sc.pp.neighbors(tmp)
    sc.tl.leiden(tmp, flavor="igraph", n_iterations=2, directed=False)

    return tmp.obs["leiden"]


adata.obs["features_cluster"] = cluster_features(adata.obsm["features"], like="summary")
print("Image clusters:", adata.obs["features_cluster"].nunique())

In [ ]:
sq.pl.spatial_scatter(adata, color=["features_cluster", "cluster"])
plt.savefig("visium_hne_image_vs_gene_clusters.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: visium_hne_image_vs_gene_clusters.png")

## 6. Build Spatial Graph & Neighborhood Enrichment

**Neighborhood enrichment** answers: which cluster pairs are spatially adjacent more often than expected by chance?

Method: Build a spatial connectivity graph → count adjacencies for each cluster pair → compare to 1000 random permutations.

> **Save this plot**

In [ ]:
sq.gr.spatial_neighbors(adata)
print(f"Connectivity matrix: {adata.obsp['spatial_connectivities'].shape}")

In [ ]:
sq.gr.nhood_enrichment(adata, cluster_key="cluster")
sq.pl.nhood_enrichment(adata, cluster_key="cluster")
plt.savefig("visium_hne_nhood_enrichment.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: visium_hne_nhood_enrichment.png")

## 7. Co-occurrence Across Spatial Dimensions

Co-occurrence measures how the probability of observing cluster B changes as we increase the radius around cluster A.

$$\text{Score} = \frac{p(\text{exp} \mid \text{cond})}{p(\text{exp})}$$

Score > 1 means enriched spatial co-occurrence at that radius. We examine the Hippocampus cluster.

> **Save this plot**

In [ ]:
sq.gr.co_occurrence(adata, cluster_key="cluster")
sq.pl.co_occurrence(
    adata,
    cluster_key="cluster",
    clusters="Hippocampus",
    figsize=(8, 4),
)
plt.savefig("visium_hne_co_occurrence.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: visium_hne_co_occurrence.png")

## 8. Ligand-Receptor Interaction Analysis

After finding co-occurring clusters, we ask: what molecular signals could drive these interactions? Squidpy re-implements CellPhoneDB with the OmniPath database.

We focus on Hippocampus → Pyramidal_layer communication, filtering for high mean expression and low adjusted p-value.

> **Save this plot**

In [ ]:
sq.gr.ligrec(adata, n_perms=100, cluster_key="cluster")
print("Ligand-receptor analysis complete.")

In [ ]:
sq.pl.ligrec(
    adata,
    cluster_key="cluster",
    source_groups="Hippocampus",
    target_groups=["Pyramidal_layer", "Pyramidal_layer_dentate_gyrus"],
    means_range=(3, np.inf),
    alpha=1e-4,
    swap_axes=True,
)
plt.savefig("visium_hne_ligrec.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: visium_hne_ligrec.png")

## 9. Spatially Variable Genes — Moran's I

Moran's I measures spatial autocorrelation of gene expression:
- **I near +1**: expression spatially clustered (spatially variable gene)
- **I near 0**: random spatial pattern

We test the top 1000 highly variable genes.

> **Save this plot**

In [ ]:
genes = adata[:, adata.var.highly_variable].var_names.values[:1000]
print(f"Testing {len(genes)} genes for spatial autocorrelation...")

sq.gr.spatial_autocorr(
    adata,
    mode="moran",
    genes=genes,
    n_perms=100,
    n_jobs=1,
)

print("\nTop 10 spatially variable genes:")
print(adata.uns["moranI"].head(10))

In [ ]:
top_genes = adata.uns["moranI"].head(3).index.tolist()
print(f"Top spatially variable genes: {top_genes}")

sq.pl.spatial_scatter(adata, color=top_genes + ["cluster"])
plt.savefig("visium_hne_spatially_variable_genes.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: visium_hne_spatially_variable_genes.png")

## 10. Summary

| Analysis | Key Finding |
|---|---|
| Image vs gene clusters | Fiber_tract and Hippocampus broadly matched; cortex differs |
| Neighborhood enrichment | Pyramidal_layer highly enriched near Hippocampus |
| Co-occurrence | Pyramidal layers co-occur at short distances with Hippocampus |
| Ligand-receptor | Candidate molecular drivers of Hippocampus-Pyramidal communication |
| Moran's I | Top genes (Olfm1, Plp1, Itpka) show strong spatial clustering |

**Next:** `04_xenium_analysis.ipynb` — single-cell resolution spatial data.

In [ ]:
adata.write_h5ad("visium_hne_processed.h5ad")
print("Saved: visium_hne_processed.h5ad")
print("\nFigures to download:")
for f in [
    "visium_hne_cluster_spatial.png",
    "visium_hne_image_vs_gene_clusters.png",
    "visium_hne_nhood_enrichment.png",
    "visium_hne_co_occurrence.png",
    "visium_hne_ligrec.png",
    "visium_hne_spatially_variable_genes.png",
]:
    print(f"  - {f}")